In [ ]:
# 1. Data Loading and Exploration
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

CSV_PATH = "train.csv"  # Mobile Price Classification train dataset
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nHead:\n", df.head())
print("\nDtypes:\n", df.dtypes)
target_col = "price_range" if "price_range" in df.columns else None
features = [c for c in df.columns if c != target_col] if target_col else df.columns.tolist()

print("\nBasic describe (numeric):\n", df.select_dtypes(include=[np.number]).describe())

# 2. Data Cleaning and Preprocessing
# handle missing
df = df.replace([np.inf, -np.inf], np.nan)
missing_before = df.isna().sum()
df = df.fillna(df.median(numeric_only=True))
df = df.fillna("Unknown")

# categorical -> numeric
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# 3. Statistical Analysis with NumPy and SciPy
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col and target_col in num_cols:
    num_cols = [c for c in num_cols if c != target_col]

# central tendency and variability
summary = pd.DataFrame({
    "mean": df[num_cols].mean(),
    "median": df[num_cols].median(),
    "mode": df[num_cols].mode().iloc[0],
    "min": df[num_cols].min(),
    "max": df[num_cols].max(),
    "range": df[num_cols].max() - df[num_cols].min(),
    "var": df[num_cols].var(ddof=1),
    "std": df[num_cols].std(ddof=1),
    "skew": df[num_cols].skew(),
    "kurtosis": df[num_cols].kurtosis()
})
print("\nStats summary:\n", summary)

# hypothesis testing between groups (e.g., price_range 0 vs 3 if available)
if target_col and df[target_col].nunique() >= 2:
    groups = sorted(df[target_col].unique())
    g_min, g_max = groups[0], groups[-1]
    feat_tests = {}
    for col in num_cols:
        a = df.loc[df[target_col] == g_min, col].dropna()
        b = df.loc[df[target_col] == g_max, col].dropna()
        if len(a) > 10 and len(b) > 10:
            t, p = stats.ttest_ind(a, b, equal_var=False)
            feat_tests[col] = {"t": t, "p": p}
    sig = {k: v for k, v in feat_tests.items() if v["p"] < 0.05}
    print(f"\nT-tests ({target_col}={g_min} vs {g_max}) significant (p<0.05):", list(sig.keys())[:15])

# feature-target correlations (Spearman robust to monotonic rel.)
if target_col:
    corr_results = {}
    for col in num_cols:
        rho, p = stats.spearmanr(df[col], df[target_col])
        corr_results[col] = (rho, p)
    corr_sorted = sorted(corr_results.items(), key=lambda x: abs(x[1][0]), reverse=True)[:10]
    print("\nTop 10 |Spearman rho| with target:\n", corr_sorted)

# 4. Data Visualization with Matplotlib
# histograms
for col in num_cols[:6]:
    plt.hist(df[col].dropna(), bins=30)
    plt.title(f"Histogram - {col}")
    plt.xlabel(col); plt.ylabel("Count")
    plt.tight_layout(); plt.show()

# scatter vs target (for numeric target not categorical skip)
if target_col and df[target_col].nunique() > 4:
    for col in num_cols[:4]:
        plt.scatter(df[col], df[target_col], s=8, alpha=0.5)
        plt.title(f"{col} vs {target_col}")
        plt.xlabel(col); plt.ylabel(target_col)
        plt.tight_layout(); plt.show()

# boxplots per class if target exists
if target_col:
    for col in num_cols[:6]:
        sns.boxplot(x=df[target_col], y=df[col])
        plt.title(f"Boxplot - {col} by {target_col}")
        plt.tight_layout(); plt.show()

# correlation heatmap
corr = df[num_cols + ([target_col] if target_col else [])].corr(method="spearman")
sns.heatmap(corr, cmap="viridis")
plt.title("Spearman Correlation Heatmap")
plt.tight_layout(); plt.show()

# 5. Insight Synthesis and Conclusion
# print simple insights
if target_col:
    print("\nClass balance:\n", df[target_col].value_counts().sort_index())
    top_feats = [name for name, (rho, p) in corr_sorted[:5]] if target_col else []
    print("\nPotential key determinants (by |Spearman rho|):", top_feats)

print("\nNotes:")
print("- Missing values handled with median for numeric and placeholder for categorical.")
print("- T-tests compare extreme classes; ANOVA can be added for multi-class comparisons.")
print("- Visual patterns and significant tests highlight candidate features for classification.")
